# **Generator–Evaluator (Feedback Loop)**

The **generator** creates a joke. The **evaluator** rates it and gives feedback.
If the joke isn't funny (and we haven't hit the iteration cap), the loop repeats.

Flow: `START → generate_node → (check_iteration) → evaluate_node → generate_node → ...`

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv
import os

load_dotenv()

if os.environ.get('GROQ_API_KEY'):
    print("Groq API Key is set.")
else:
    raise ValueError("GROQ_API_KEY not found.")

In [ ]:
llm = ChatGroq(model="llama3-8b-8192", temperature=0)

### **Pydantic Schema for the Evaluator**
`funny_flag` is restricted to exactly two values via `Literal` — prevents ambiguous outputs.

In [ ]:
from pydantic import BaseModel, Field
from typing import List, TypedDict, Literal

class llm_schema(BaseModel):
    funny_flag: Literal["funny", "not funny"] = Field(description="Whether the joke is funny or not")
    feedback:   str                            = Field(description="Feedback on the joke, if not funny")

llm_with_schema = llm.with_structured_output(llm_schema)

### **Graph Schema**

In [ ]:
class graph_schema(TypedDict):
    topic:          str   # the topic for the joke
    joke:           str   # the current joke (updated each iteration)
    funny_flag:     str   # "funny" or "not funny" — set by evaluate_node
    feedback:       str   # improvement tips from evaluate_node
    max_iterations: int   # counts iterations to prevent infinite loops

### **Node Functions**

In [ ]:
def generate_node(state: graph_schema) -> graph_schema:
    """Creates or improves a joke based on feedback from the previous evaluation."""

    topic = state['topic']

    if state['feedback']:
        # Not the first iteration — incorporate the evaluator's feedback
        response = llm.invoke(
            f"Please modify the following joke '{state['joke']}' based on this feedback: {state['feedback']}"
        )
    else:
        # First iteration — generate a fresh joke from scratch
        response = llm.invoke(f"Create only one joke about the following topic: {topic}")

    state['joke'] = response.content
    return state


def evaluate_node(state: graph_schema) -> graph_schema:
    """Evaluates whether the joke is funny and records feedback + iteration count."""

    joke      = state['joke']
    iteration = state['max_iterations']

    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a comedy critic. Evaluate the following joke and provide feedback on how to make it funnier."),
        ("user",   f"Evaluate this joke: {joke}\nRespond with 'funny' or 'not funny' and provide feedback if it's not funny.")
    ])

    chain    = prompt | llm_with_schema
    response = chain.invoke({"joke": joke})

    state['funny_flag']     = response.funny_flag
    state['feedback']       = response.feedback
    state['max_iterations'] = iteration + 1   # increment the counter

    return state

### **Conditional Edge Function**
Decides whether to loop back for another try or exit.

In [ ]:
def check_iteration(state: graph_schema) -> str:
    """If we're within the iteration limit AND the joke isn't funny, evaluate it. Otherwise end."""

    iteration = state['max_iterations']

    if iteration <= 5 and state['funny_flag'] != "funny":
        return "evaluate_node"   # keep looping
    else:
        return "end"             # exit — either funny enough or max iterations reached

### **Build the Graph**

In [ ]:
from langgraph.graph import StateGraph, START, END
from IPython.display import Image

graph = StateGraph(graph_schema)

graph.add_node("generate_node", generate_node)
graph.add_node("evaluate_node", evaluate_node)

graph.add_edge(START, "generate_node")

# After generate_node, call check_iteration to decide the next step
graph.add_conditional_edges(
    "generate_node",
    check_iteration,
    {"evaluate_node": "evaluate_node", "end": END}
)

# After evaluate_node, always loop back to generate_node
graph.add_edge("evaluate_node", "generate_node")
graph.add_edge("generate_node", END)

evaluator_graph = graph.compile()
Image(evaluator_graph.get_graph().draw_mermaid_png())

### **Stream the Feedback Loop**

In [ ]:
for chunk in evaluator_graph.stream(
    {
        "topic":          "Cars",
        "joke":           "",
        "funny_flag":     "",
        "feedback":       "",
        "max_iterations": 0
    },
    stream_mode="updates"
):
    print(chunk)   # prints state updates after each node — you can watch the loop in real time